# Evidence Agent with mandatory human review

This notebook demonstrates source-linked evidence discovery for a generic virus project. API results become review-pending proposals only. They do not alter gene scores, essentiality calls, or curated evidence until a researcher explicitly approves them. Expression is not essentiality, ortholog evidence is not direct target-virus evidence, and missing search results are not evidence of non-essentiality.

In [ ]:
from pathlib import Path
import pandas as pd
import yaml

cwd = Path.cwd().resolve()
ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
PROJECT = ROOT / 'configs/projects/hsv2_case_study.yaml'
OUTPUT = ROOT / 'reports/hsv2_evidence_agent'
assert PROJECT.is_file(), PROJECT

## 1. Build the annotation-derived gene and alias catalog

Nothing below is HSV-gene-name logic. Gene names, locus tags, products and protein identifiers are read from the configured GFF.

In [ ]:
from viral_safe_target.evidence_agent import build_gene_catalog, build_search_queries
from viral_safe_target.project_workflow import load_project

context = load_project(PROJECT)
gff = context.profiles.resolve(context.profiles.virus['annotation_gff'])
gff_available = bool(gff and gff.is_file())
public_catalog = OUTPUT / 'gene_catalog.tsv'
if gff_available:
    gene_catalog = build_gene_catalog(gff, context.profiles.virus)
elif public_catalog.is_file():
    gene_catalog = pd.read_csv(public_catalog, sep='\t', dtype=str).fillna('')
else:
    raise FileNotFoundError(f'Neither annotation GFF nor public gene catalog is available: {gff}; {public_catalog}')
search_queries = build_search_queries(gene_catalog, context.profiles.virus)
gene_catalog.head(10)

In [ ]:
query_summary = search_queries.groupby(['evidence_scope', 'query_family']).size().rename('query_count').reset_index()
query_summary

## 2. Optional official-API discovery

Network execution is off by default so the notebook remains deterministic. Set `RUN_NETWORK=True` to query PubMed/NCBI E-utilities, Europe PMC, UniProt and NCBI reference metadata. Responses are cached with request and content checksums.

In [ ]:
from viral_safe_target.evidence_agent import discover_evidence

RUN_NETWORK = False
FOCUS_GENES = ['UL3', 'UL10', 'UL18', 'UL20', 'UL36', 'UL52', 'UL53', 'UL19', 'UL30']
if RUN_NETWORK and not gff_available:
    raise FileNotFoundError('Network discovery requires the project annotation GFF; the public catalog supports offline review only.')
if RUN_NETWORK:
    discovery = discover_evidence(
        gff_path=gff,
        virus_profile=context.profiles.virus,
        out_dir=OUTPUT,
        maximum_results_per_query=5,
        genes=FOCUS_GENES,
    )
else:
    discovery = {'mode': 'network disabled', 'output_dir': str(OUTPUT)}
discovery

## 3. Inspect the review queue

Each row links to its source, records direct/ortholog/unresolved scope, identifies the proposed experiment class, and contains a short source span. The proposal remains pending.

In [ ]:
queue_path = OUTPUT / 'review_queue.tsv'
review_queue = pd.read_csv(queue_path, sep='\t', dtype=str).fillna('') if queue_path.is_file() else pd.DataFrame()
review_queue.head(20)

In [ ]:
if review_queue.empty:
    review_checks = {'proposal_count': 0, 'all_pending': True, 'automatic_approval': False}
else:
    review_checks = {
        'proposal_count': len(review_queue),
        'genes_with_proposals': review_queue['gene_name'].nunique(),
        'all_pending': review_queue['review_status'].eq('pending').all(),
        'direct_target_rows': int(review_queue['evidence_scope'].eq('direct_target_virus').sum()),
        'ortholog_rows': int(review_queue['evidence_scope'].eq('ortholog').sum()),
        'unknown_essentiality_rows': int(review_queue['proposed_essentiality_call'].eq('unknown').sum()),
        'automatic_approval': False,
    }
review_checks

## 4. Researcher decision boundary

Open `review_queue.tsv`, read the linked source and surrounding context, correct any proposed fields, then choose `approved`, `rejected`, or `needs_revision`. Approved rows require reviewer identity and review date. Only then run:

```bash
vst evidence apply --project configs/projects/hsv2_case_study.yaml \
  --review-queue reports/hsv2_evidence_agent/review_queue.tsv
```

The apply command exports approved rows only. It preserves evidence-virus scope and cannot convert HSV-1 evidence into direct HSV-2 evidence. No wet-lab procedure or therapeutic conclusion is produced.